# log-back — ex3: log_back at the domain edge — matches torch.autograd

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `log-back`. Running the final beacon cell reports progress against the `Backprop: log_back` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: log_back` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`log-back`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "log-back"
DD_SUBTOPIC = "Backprop: log_back"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## log_back at the domain edge — `1/x` blows up near zero

Ex1 derived `log_back(grad_out, out, x) = grad_out / x` and ex2 composed it with multiply_back. The deepening move exercises the DOMAIN BEHAVIOUR: `log(x)` is only defined for `x > 0`, and `log_back` inherits a `1/x` singularity at the origin.

```
x = 1.0          ->  log_back(1, _, 1)    = 1.0           (tame)
x = 1e-10        ->  log_back(1, _, 1e-10) = 1e10          (large)
x = 1e-30        ->  log_back(1, _, 1e-30) = 1e30          (huge — float overflow nearby)
x = 0.0          ->  log_back(1, _, 0)    = +inf           (division)
x < 0            ->  forward log(x) was already nan; back is nan too
```

**The drill verifies our hand-rolled `log_back` matches `torch.autograd` BIT-FOR-BIT on the same inputs** — even at the blow-up. That's the test: real autograd doesn't 'protect' you from `1/x`; it just computes it. If your model overflows at log of a near-zero input, the fix is upstream (clamp, eps-add), not in the back-fn.

**Why no `eps` in log_back.** Some libraries silently add `+ 1e-8` to the denominator. PyTorch doesn't — and our drill doesn't either. Correctness = matching the closed-form gradient exactly; numerical stability is the caller's job.

### Exercise 3 — log_back at the domain edge — matches torch.autograd

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze log_back's behaviour at small-x and zero-x inputs by computing it directly and comparing bit-for-bit against torch.autograd, confirming neither implementation adds an eps or clamps — both reproduce the 1/x blow-up faithfully.
> Keywords: log-back, domain, blowup, torch-autograd, comparison
> ```

**KCs targeted:** `log-back-no-eps-no-clamp`, `matches-torch-autograd-at-edge`

Implement `log_back(grad_out, out, x)` AND a comparison helper `ex3_compare_with_torch(x_values, grad_out_values)` that returns the side-by-side audit of hand-rolled `log_back` against `torch.autograd`.

Function 1: `log_back(grad_out, out, x)` — same as ex1: `return grad_out / x`. The `out` argument is unused (we keep it in the signature for dispatcher compatibility).

Function 2: `ex3_compare_with_torch(x_values, grad_out_values)` where both inputs are 1-D `torch.Tensor` of the same length. For EACH index `i`:

1. Compute `hand = log_back(grad_out_values[i:i+1], None, x_values[i:i+1])` (single-element slice).
2. Compute the torch.autograd answer by building `x_i = x_values[i:i+1].clone().requires_grad_(True)`, then `loss = (t.log(x_i) * grad_out_values[i:i+1]).sum()`, then `loss.backward()`, and reading `x_i.grad`.
3. Append `(hand.item(), torch_answer.item())` to a result list.

Return the list of `(hand_value, torch_value)` pairs, one per input element.

Constraints:
- Don't add an `eps`. Don't `.clamp_min`. Don't `torch.where`.
- The test will pass values like `1e-30`, `0.0` (the test handles the inf case carefully), and large values, and expects your answers to MATCH `torch.autograd` exactly (or in the inf case, both to be `inf`).

In [ ]:
def log_back(grad_out, out, x):
    """Elementwise backward for out = log(x). Returns grad_out / x."""
    raise NotImplementedError()


def ex3_compare_with_torch(x_values, grad_out_values):
    """Return list of (hand, torch) gradient pairs for each input."""
    raise NotImplementedError()


def _test_ex3():
    # --- log_back smoke test (tame values) ---
    x = t.tensor([1.0, 2.0, 4.0, 8.0])
    g = log_back(t.ones(4), None, x)
    assert t.allclose(g, 1.0 / x, atol=1e-7), f'log_back basic wrong: {g}'

    # --- comparison helper: tame inputs all agree to ~machine precision ---
    x_tame = t.tensor([1.0, 2.0, 10.0, 100.0])
    go_tame = t.tensor([1.0, 1.0, 1.0, 1.0])
    pairs = ex3_compare_with_torch(x_tame, go_tame)
    assert len(pairs) == 4, f'one pair per input; got {len(pairs)}'
    for i, (hand, torch_ans) in enumerate(pairs):
        assert abs(hand - torch_ans) < 1e-6, (
            f'index {i}: hand={hand} torch={torch_ans} (tame range)'
        )

    # --- domain edge: very small x. hand-rolled blows up; torch blows up the same way ---
    x_small = t.tensor([1e-3, 1e-6, 1e-10, 1e-20])
    go_small = t.tensor([1.0, 1.0, 1.0, 1.0])
    pairs = ex3_compare_with_torch(x_small, go_small)
    for i, (hand, torch_ans) in enumerate(pairs):
        # Compare in relative terms — values may be huge.
        expected = 1.0 / x_small[i].item()
        rel = abs(hand - torch_ans) / max(abs(torch_ans), 1e-30)
        assert rel < 1e-5, (
            f'small-x mismatch at i={i}: hand={hand} torch={torch_ans} expected~{expected}'
        )
        # And both should match the 1/x ground truth.
        rel_hand = abs(hand - expected) / max(abs(expected), 1e-30)
        assert rel_hand < 1e-5, f'hand-rolled missed 1/x at i={i}: {hand} vs {expected}'

    # --- huge x: small grad ---
    x_big = t.tensor([1e10, 1e15])
    go_big = t.tensor([1.0, 1.0])
    pairs = ex3_compare_with_torch(x_big, go_big)
    for i, (hand, torch_ans) in enumerate(pairs):
        assert abs(hand - torch_ans) < 1e-25 or (hand == 0.0 and torch_ans == 0.0), (
            f'big-x mismatch at i={i}: hand={hand} torch={torch_ans}'
        )

    # --- non-unit grad_out: scales the answer linearly ---
    x_mix = t.tensor([2.0, 4.0])
    go_mix = t.tensor([7.0, -3.0])
    pairs = ex3_compare_with_torch(x_mix, go_mix)
    for i, (hand, torch_ans) in enumerate(pairs):
        expected = go_mix[i].item() / x_mix[i].item()
        assert abs(hand - expected) < 1e-6, f'hand wrong: {hand} vs {expected}'
        assert abs(torch_ans - expected) < 1e-6

    # --- x=0: both implementations produce inf (matching). We check sign for grad_out=1.0. ---
    x_zero = t.tensor([0.0])
    go_zero = t.tensor([1.0])
    pairs = ex3_compare_with_torch(x_zero, go_zero)
    hand, torch_ans = pairs[0]
    import math
    # Both should be +inf (1/0 with positive numerator).
    assert math.isinf(hand), f'hand at x=0 should be inf; got {hand}'
    assert math.isinf(torch_ans), f'torch at x=0 should be inf; got {torch_ans}'
    assert (hand > 0) == (torch_ans > 0), f'sign should agree; hand={hand} torch={torch_ans}'

    # --- shape: helper returns Python list, each pair Python (float, float) ---
    pairs = ex3_compare_with_torch(t.tensor([1.0, 2.0]), t.tensor([1.0, 1.0]))
    assert isinstance(pairs, list)
    assert all(isinstance(p, tuple) and len(p) == 2 for p in pairs)
    assert all(isinstance(v, float) for p in pairs for v in p)
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def log_back(grad_out, out, x):
    return grad_out / x


def ex3_compare_with_torch(x_values, grad_out_values):
    pairs = []
    for i in range(x_values.numel()):
        x_slice = x_values[i:i+1]
        go_slice = grad_out_values[i:i+1]
        hand = log_back(go_slice, None, x_slice).item()
        x_var = x_slice.clone().detach().requires_grad_(True)
        loss = (t.log(x_var) * go_slice).sum()
        loss.backward()
        torch_ans = x_var.grad.item()
        pairs.append((hand, torch_ans))
    return pairs
```

**No eps, no clamp.** Real `torch.autograd` for `log` does NOT add a stability term. If you want eps, you add it to the FORWARD (`log(x + eps)`), which changes the derivative to `1/(x+eps)` automatically via the chain rule. Doing it inside `log_back` would silently disagree with the forward — autograd's first correctness rule is that backward matches forward's analytic derivative.

**Why `.item()` for comparison.** The pairs are pairs of Python floats. At inf, two tensors comparing via `==` work but lose the is-it-actually-inf signal. `math.isinf` is the precise check.

**Per-element loop, not vectorised.** We unroll because `torch.autograd` accumulates `.grad` into the leaf, so running it vectorised would mix gradients across positions. The slice + `requires_grad_(True)` per element keeps each backward isolated. Slower but correct; this is a comparison drill, not a perf drill.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()